In [1]:
%run Functions_NCS-2-9-26.ipynb

In [2]:
import random
import json
from datetime import datetime
import numpy as np
import pandas as pd
import os #move the current working directory
import matplotlib.pyplot as plt
import glob

# The goal of this workbook is to read in ellipsometry data and the corresponding JSON files in order to get the "input data" (SE data), and "output data" (Layers, Thicknesses, Compositions...) to train a neural network.


In [3]:
path = r'C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data'

csv_file = glob.glob(path + "\*.csv")

json_file = glob.glob(path + "\*.json")

In [4]:
for i in range(len(csv_file)):
    
    print(json_file[i])

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.412577.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.447968.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.480440.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.503874.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.544376.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.568478.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.600042.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.642069.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.670818.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Traini

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.465638.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.489682.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.521759.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.560879.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.594798.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.635492.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.667238.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.701874.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-24-18.732319.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Traini

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.685506.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.713184.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.748711.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.788689.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.837702.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.877520.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.915182.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.940263.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-02.996438.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Traini

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.300680.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.344488.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.372652.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.396985.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.436453.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.468692.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.507350.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.533893.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-25-42.565533.json
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Traini

In [5]:
def read_SE_data(path):

    '''
    This function will need a folder path as an input and will produce a set of SE data along with the information needed to generate the SE data
    The folder is expected to contain csv files containing the SE data and the JSON files containing the information needed to simulated the SE data.

    '''
    
    #Step 1: read in csv and json files
    
    csv_file = glob.glob(path + "\*.csv")

    json_file = glob.glob(path + "\*.json")
    
    #Step 1.5: actually read it
    
    SE_data = [] #list to store the SE data
    Metadata = [] #list to store the meta data
    L1_Thickness = []
    L2_Thickness = []
    L1_Composition = []
    
    for i in range(len(csv_file)):
    
        print(csv_file[i])
        
        df1 = pd.read_csv(csv_file[i]) #read in CSV files
        df1.drop(df1.columns[0], axis=1, inplace=True) #remove the first column because there is extra unneeded data
        df1.drop(df1.columns[0], axis=1, inplace=True) #Remove wavelength column
        SE_data.append(df1) #save SE data
        #SE_data = np.array(SE_data)
        #print(np.shape(SE_data))
    
        df2 = pd.read_json(json_file[i])
        #Metadata.append(df2) #Optionally save json file
        
        #read composition
        Composition = float(df2['Layer Info']['Layer_1']['Tellerium_Content(%)'])
        
        
        #read thicknesses
        Thickness1 = float(df2['Layer Info']['Layer_1']['thickness_nm'])
        Thickness2 = float(df2['Layer Info']['Layer_2']['thickness_nm'])
        L1_Thickness.append(Thickness1)
        L2_Thickness.append(Thickness2)
        
        
        #convert composition to binary categorical data
        
        if Composition == 86: 
            arr = np.array([1,0,0, #Convert 86 to binary catergory
                            0,0,0,
                            0,0,0])
            L1_Composition.append(arr)
        if Composition == 75: 
            arr = np.array([0,1,0, #Convert 75 to binary catergory
                            0,0,0,
                            0,0,0])
            L1_Composition.append(arr)
        if Composition == 69:
            arr = np.array([0,0,1, #Convert 69 to binary catergory
                            0,0,0,
                            0,0,0])
            L1_Composition.append(arr)
        if Composition == 64:
            arr = np.array([0,0,0, #Convert 64 to binary catergory
                            1,0,0,
                            0,0,0])
            L1_Composition.append(arr)
        if Composition == 54:
            arr = np.array([0,0,0, #Convert 54 to binary catergory
                            0,1,0,
                            0,0,0])
            L1_Composition.append(arr)
        if Composition == 38:
            arr = np.array([0,0,0, #Convert 38 to binary catergory
                            0,0,1,
                            0,0,0])
            L1_Composition.append(arr)
        if Composition == 25:
            arr = np.array([0,0,0, #Convert 25 to binary catergory
                            0,0,0,
                            1,0,0])
            L1_Composition.append(arr)
        if Composition == 15:
            arr = np.array([0,0,0, #Convert 15 to binary catergory
                            0,0,0,
                            0,1,0])
            L1_Composition.append(arr)
        if Composition == 0:
            arr = np.array([0,0,0, #Convert 0 to binary catergory
                            0,0,0,
                            0,0,1])
            L1_Composition.append(arr)
    
    
    #Put Se data into an array
    
    SE_data = np.array(SE_data)
    
    #Put thickness and Composition into an array
    
    L1_Thickness = np.array(L1_Thickness)
    L2_Thickness = np.array(L2_Thickness)
    L1_Composition = np.array(L1_Composition)
    
    #output, numpy array of 1 column that is data, (number of simulations, 3(NCS), # of points), 
    return(SE_data, L1_Thickness, L2_Thickness, L1_Composition)

In [6]:
def unison_shuffled_copies(*arrays):
    # Ensure all input arrays have the same length
    lengths = [len(arr) for arr in arrays]
    assert all(length == lengths[0] for length in lengths), "All arrays must have the same length"

#Generate a random permutation of indices
    p = np.random.permutation(lengths[0])

#Shuffle each array according to the permutation
    return tuple(arr[p] for arr in arrays)

In [7]:
path = r'C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data' #Define file path
#Load in SE data, thicknesses, and composition
SE_data_train, L1_Thickness_train, L2_Thickness_train, L1_Composition_train = read_SE_data(path)

#SE_data = np.array(SE_data)
#Randomize
SE_data_train, L1_Thickness_train, L2_Thickness_train, L1_Composition_train=unison_shuffled_copies(SE_data_train, L1_Thickness_train, L2_Thickness_train, L1_Composition_train)

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.412577.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.447968.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.480440.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.503874.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.544376.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.568478.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.600042.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.642069.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-24.670818.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.508975.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.546161.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.591264.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.624261.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.647561.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.671708.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.711896.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.735088.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-27.766613.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.597411.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.630487.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.672675.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.703324.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.735937.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.767954.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.800856.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.833649.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-30.866504.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.494353.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.526949.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.561237.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.592284.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.625213.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.657871.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.690595.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.725411.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-33.755865.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.767701.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.813319.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.845883.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.894417.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.922799.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.955309.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-36.998858.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-37.030976.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-23-37.063266.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

...

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.241175.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.279674.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.303526.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.360380.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.382770.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.414061.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.445492.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.476897.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-04.515773.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.084049.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.115933.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.147748.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.179768.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.211858.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.243779.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.273742.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.310767.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-07.334875.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-09.962363.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-09.994061.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.026448.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.060267.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.089786.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.121315.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.153470.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.194672.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-10.217164.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.784225.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.805834.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.847822.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.876743.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.900984.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.942675.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.974791.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-12.997280.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-13.029538.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.582632.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.622769.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.656631.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.681061.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.713382.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.745290.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.786009.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.809925.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2026-02-24_13-26-15.851661.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Training_Data\2

In [8]:
path = r'C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data' #Define file path
#Load in SE data, thicknesses, and composition
SE_data_val, L1_Thickness_val, L2_Thickness_val, L1_Composition_val = read_SE_data(path)

#SE_data = np.array(SE_data)
#Randomize
SE_data_val, L1_Thickness_val, L2_Thickness_val, L1_Composition_val=unison_shuffled_copies(SE_data_val, L1_Thickness_val, L2_Thickness_val, L1_Composition_val)

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.353219.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.387825.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.418834.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.450985.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.480826.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.529942.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.557536.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.589237.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-39.622966.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.440631.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.472201.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.507591.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.541038.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.564374.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.596174.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.634233.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.666693.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-42.700127.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.360043.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.411446.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.441968.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.479896.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.517735.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.538705.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.572097.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.601499.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-45.633013.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-47.977351.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.013941.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.045789.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.087500.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.113534.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.143095.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.179427.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.204056.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-48.235711.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-50.920088.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-50.954219.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-50.992211.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-51.015896.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-51.047872.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-51.081829.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-51.111715.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-51.143751.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-51.175962.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

...

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.530884.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.562603.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.601689.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.631513.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.665308.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.697787.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.741453.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.776314.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-56.799251.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.141588.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.176695.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.202823.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.236307.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.266304.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.298172.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.329977.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.370040.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-27-59.401457.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-01.869645.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-01.896799.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-01.931166.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-01.964369.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-01.988397.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-02.022055.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-02.052024.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-02.095125.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-02.135284.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.698010.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.730590.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.764556.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.794190.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.826114.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.871833.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.921672.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.946768.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-04.985989.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.489606.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.514634.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.561977.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.613002.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.641199.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.679417.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.709825.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.744129.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Validation_Data\2026-02-24_13-28-07.776871.csv
C:\Users\evara\OneDrive\Documents\Toledo\Machine Learni

In [ ]:
L1_Composition[7]

In [ ]:
path = r'C:\Users\evara\OneDrive\Documents\Toledo\Machine Learning\Test_Data' #Define file path
#Load in SE data, thicknesses, and composition
SE_data_test, L1_Thickness_test, L2_Thickness_test, L1_Composition_test = read_SE_data(path)

#SE_data = np.array(SE_data)
#Randomize
SE_data_test, L1_Thickness_test, L2_Thickness_test, L1_Composition_test=unison_shuffled_copies(SE_data_test, L1_Thickness_test, L2_Thickness_test, L1_Composition_test)

In [ ]:
i=1
print(L2_Thickness[i])
print(L1_Composition[i])

In [ ]:
#SE_data[0]
#L1_Thickness[0]
print('SE_data Shape', SE_data_test.shape)
print('L1 Thickness Shape', L1_Thickness.shape)
print('L2 Thickness Shape', L2_Thickness.shape)
print('L1 Composition Shape', L1_Composition.shape)

In [18]:
pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install keras

Note: you may need to restart the kernel to use updated packages.


In [16]:
pip install numpy==1.22

  Attempting uninstall: numpy
    Found existing installation: numpy 1.21.0
    Uninstalling numpy-1.21.0:
      Successfully uninstalled numpy-1.21.0



In [ ]:
import tensorflow as tf
from tensorflow import keras
import tensorflow.keras.layers as tfl

In [ ]:
#start with a composition guesser
x_train = SE_data_train #ellipsometry data
y_train = L1_Composition_train # y is the composition

x_val = SE_data_val 
y_val = L1_Composition_val

In [ ]:
'''
Dense layers
Define a series of nodes referred to as neurons, have some input data, with some shape, (697,3)=2100, each data point will connect with however many nodes we have, dense 6, every single one of those points connects with the neurons, the weights being trained is the connecting neurons, 6*2100 weights, the connections on the layer, you can add another layer, say 3, then each of the 6 will connect with the 3 next nodes, make 18 more connections, 
Input layer sections before the weight, however many nodes section is the hidden layer, what we know, the output layer, our output here will be 9
we should know input and output, and we need to find the hidden layer 
weights have certain numbers with relationships, functions scalled activation functions (Relu is one of them)
Relu -- x-axis is the input, y-axis is the value it gives the next part,
below y=0, the x is all 0, above x=0, it is a positive linear plot, 
This activation function with no bound
Each one has its own Relu,
output, when one is one, the rest are 0 since we have a category. Softmax is another activation function, asymptotically approaches 0, height is shared between each of the output nodes,all will sum to 1, the likelyhood one is true, 

Flatten it, 697,3 flattens it to 2100, one long string, need to do it for dense layers, could be different for other shapes
need to define how many neurons you have, want to compress as you go, and output to be the smallest thing 
start with the biggest amount and funnel down to a specific feature, the values of the output layers are called features, what we are trying to get the machine learning model to learn
can use the same name, as long as it is a string, if we have a branch, then need to have new names
leaky Relu can use negative values since NCS can be negative 
define output layer, Softmax which is good for [0,1]
'''

In [18]:
#Build model

#Define the input layer of the ANN,
input_shape = (697,3) # 697 data points with the 3 columns being (N,C,S)
input_NCS = tf.keras.Input(shape=input_shape)
#flatten
x = tfl.Flatten()(input_NCS)

x=tfl.Dense(units=128,activation='leaky_relu')(x)
x=tfl.Dense(units=64,activation='leaky_relu')(x)
x=tfl.Dense(units=32,activation='leaky_relu')(x)
x=tfl.Dense(units=16,activation='leaky_relu')(x)
#now define the output layer, The softmax activation function is chosen because it is bound between [0,1]. Perfect for binary categorical problems.
x=tfl.Dense(units=9,activation='softmax')(x)

#Create the model
model=tf.keras.Model(inputs=input_NCS,outputs=x)


In [19]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 697, 3)]          0         
                                                                 
 flatten_1 (Flatten)         (None, 2091)              0         
                                                                 
 dense_5 (Dense)             (None, 128)               267776    
                                                                 
 dense_6 (Dense)             (None, 64)                8256      
                                                                 
 dense_7 (Dense)             (None, 32)                2080      
                                                                 
 dense_8 (Dense)             (None, 16)                528       
                                                                 
 dense_9 (Dense)             (None, 9)                 153   

In [22]:
#We made a model, need to compile
#To optimize it, use Adam (thing that does the gradient descent, how does it adjust the parameters), Adam is a well known one, the number is the learning rate (1e-3)
#Could pass a function, that decreases the learning rate, but for now jsut give it a number
#loss function, take your output layer of the machine learning, find your loss, this function is important, varies based on the data, categorical data has a pass-fail, numerical data is different, how far away from the center we are
#categorical cross entropy, 
#binary coross entropy, define the loss function, can do it for the whole thing, or can do it for each output, which we will need once doing the thickness checker
model.compile(
    optimizer = tf.keras.optimizers.Adam(1e-5),
    loss = tf.keras.losses.CategoricalCrossentropy()
)

#Train the model, Output how the loss changes with each generation, history of the model changing
history = model.fit(
    x=x_train, #input data
    y=y_train, #output data
    batch_size =32, #batch size, how many trials it takes
    epochs=50, #epoch, how many times it goes thru the training data set, 
    validation_data=(x_val,y_val), #validation data
    verbose=1 #verbosity tell you if it is good
)

# we can make something that looks at the validation perfomrance as we train, and once validation perfomance is stagnent, itll reduce the learning rate, and if it is still stagnent, then it will stop


Epoch 1/50
158/158 [==============================] - 1s 5ms/step - loss: 0.0051 - val_loss: 0.0088
Epoch 2/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0028 - val_loss: 0.0050
Epoch 3/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0023 - val_loss: 0.0038
Epoch 4/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0020 - val_loss: 0.0031
Epoch 5/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0018 - val_loss: 0.0028
Epoch 6/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0017 - val_loss: 0.0025
Epoch 7/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0016 - val_loss: 0.0024
Epoch 8/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0014 - val_loss: 0.0022
Epoch 9/50
158/158 [==============================] - 1s 4ms/step - loss: 0.0013 - val_loss: 0.0020
Epoch 10/50
158/158 [==============================] - 1s 5ms/step - loss: 0.0013 - val_loss: 0.0019

In [23]:
#pass the test data
x_test = SE_data_test
y_test = L1_Composition_test

model.evaluate(x_test,y_test)

29/29 [==============================] - 0s 1ms/step - loss: 0.0040


0.004007568582892418

In [24]:
test_predictions = model.predict(x_test)

29/29 [==============================] - 0s 1ms/step


In [27]:
test_predictions[0]

array([0.0000000e+00, 0.0000000e+00, 0.0000000e+00, 5.7706290e-29,
       3.6460712e-17, 1.6731224e-12, 3.1562342e-07, 2.3198914e-14,
       9.9999964e-01], dtype=float32)

In [28]:
y_test[0]

array([0, 0, 0, 0, 0, 0, 0, 0, 1])

In [40]:
#Goal is to compare y_test and test_predictions and see how many trials are in agreement and how many are not in agreement
#Step 1: Identify where the 1 is in y_test and find the max value in test_predictions
#max_y_test = y_test.max()
#max_index = y_test.index(max_y_test)

Correct_trials = 0
Failed_trials = 0
for i in range(len(y_test)):
    index_y_test = np.argmax(y_test[i])
    
    index_test_predictions = np.argmax(test_predictions[i])
    
    #check to see if the maximums are in the same spot. 
    if index_y_test == index_test_predictions:
        Correct_trials = Correct_trials + 1
    else: 
        Failed_trials = Failed_trials + 1
#Step 2: 
print('Correct Trials:', Correct_trials)
print('Failed Trials:', Failed_trials)


Correct Trials: 899
Failed Trials: 1
